# FinAuto — Research Experiments and Model Selection

This is a **curated record of completed experiments**, extracted from the original 74-cell exploratory notebook. It preserves representative code and saved outputs, without rerunning experiments. Expand the code under each experiment to inspect it. For a short executable workflow, see the separate **final model notebook**.

**Important:** The research experiments used a **stratified random 70/15/15 split** (344/74/74 frauds in train/validation/test), while the simplified final notebook used a **chronological 70/15/15 split** (52 frauds in its test set). Metrics across these notebooks are **not directly comparable**. No claim is made that one is better based on these different test sets.

**Dataset:** [European credit card fraud dataset](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud): 284,807 rows, 492 frauds, 30 input features, zero missing values. **Primary metric:** Average Precision (AP), with frauds detected among the top 100 highest-risk transactions as an operational metric.


## 1. Data and research split

### Original stratified data split

<details><summary>Original experiment code (click to expand)</summary>

```python
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Class"])
y = df["Class"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

print("Training fraud:", y_train.sum())
print("Validation fraud:", y_val.sum())
print("Testing fraud:", y_test.sum())
```

</details>

**Recorded output**

```text
Training: (199364, 30)
Validation: (42721, 30)
Testing: (42722, 30)
Training fraud: 344
Validation fraud: 74
Testing fraud: 74
```

## 2. Logistic Regression baseline

### Baseline validation evaluation

<details><summary>Original experiment code (click to expand)</summary>

```python
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

best_model = grid.best_estimator_

y_prob = best_model.predict_proba(X_val)[:, 1]
y_pred = best_model.predict(X_val)

print("Average Precision:", average_precision_score(y_val, y_prob))
print("ROC-AUC:", roc_auc_score(y_val, y_prob))
print("Precision:", precision_score(y_val, y_pred, zero_division=0))
print("Recall:", recall_score(y_val, y_pred, zero_division=0))
print("F1:", f1_score(y_val, y_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))
```

</details>

**Recorded output**

```text
Average Precision: 0.6651138133170283
ROC-AUC: 0.9688774407629192
Precision: 0.3388888888888889
Recall: 0.8243243243243243
F1: 0.48031496062992124

Confusion Matrix:
[[42528   119]
 [   13    61]]
```

## 3. XGBoost baseline

### Initial XGBoost results

<details><summary>Original experiment code (click to expand)</summary>

```python
xgb_train_prob = xgb_model.predict_proba(X_train)[:, 1]
xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

xgb_train_ap = average_precision_score(y_train, xgb_train_prob)
xgb_val_ap = average_precision_score(y_val, xgb_val_prob)

print("Training AP:", round(xgb_train_ap, 4))
print("Validation AP:", round(xgb_val_ap, 4))
print("AP Gap:", round(xgb_train_ap - xgb_val_ap, 4))

print("Validation ROC-AUC:", round(
    roc_auc_score(y_val, xgb_val_prob), 4
))
```

</details>

**Recorded output**

```text
Training AP: 0.9289
Validation AP: 0.8118
AP Gap: 0.117
Validation ROC-AUC: 0.9838
```

## 4. Isolation Forest and hybrid approach

### Isolation Forest anomaly ranking

<details><summary>Original experiment code (click to expand)</summary>

```python
print("Training AP:",
      average_precision_score(y_train, iso_train_scores))

print("Validation AP:",
      average_precision_score(y_val, iso_val_scores))

print("Validation ROC-AUC:",
      roc_auc_score(y_val, iso_val_scores))
```

</details>

**Recorded output**

```text
Training AP: 0.11044938718810324
Validation AP: 0.11070777660836549
Validation ROC-AUC: 0.9463166827108018
```

### Hybrid alert rule

<details><summary>Original experiment code (click to expand)</summary>

```python
xgb_alert = xgb_val_prob >= 0.1
iso_alert = iso_val_scores >= np.percentile(normal_scores, 95)

hybrid_pred = (xgb_alert | iso_alert).astype(int)

print("Precision:", precision_score(y_val, hybrid_pred))
print("Recall:", recall_score(y_val, hybrid_pred))
print("F1:", f1_score(y_val, hybrid_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_val, hybrid_pred))
```

</details>

**Recorded output**

```text
Precision: 0.02773988176443838
Recall: 0.8243243243243243
F1: 0.05367355917289925
Confusion Matrix:
[[40509  2138]
 [   13    61]]
```

**Interpretation:** The evaluated hybrid alert rule increased false alarms substantially; anomaly scores alone did not establish fraud.

## 5. Feature engineering and tuning

### XGBoost parameter comparison

<details><summary>Original experiment code (click to expand)</summary>

```python
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

y_train_array = np.asarray(y_train)
y_val_array = np.asarray(y_val)

imbalance_ratio = (
    (y_train_array == 0).sum()
    / (y_train_array == 1).sum()
)

results = []

for depth in [3, 4, 6]:
    for weight_factor in [0.5, 1.0]:
        for min_child in [1, 5]:

            model = XGBClassifier(
                n_estimators=300,
                max_depth=depth,
                min_child_weight=min_child,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=imbalance_ratio * weight_factor,
                objective="binary:logistic",
                eval_metric="aucpr",
                random_state=42,
                n_jobs=-1
            )

            model.fit(train_fe, y_train)

            scores = model.predict_proba(val_fe)[:, 1]

            ap = average_precision_score(y_val, scores)

            top_100 = np.argsort(scores)[-100:]
            recall_100 = (
                y_val_array[top_100].sum()
                / y_val_array.sum()
            )

            results.append({
                "max_depth": depth,
                "min_child_weight": min_child,
                "weight_factor": weight_factor,
                "AP": ap,
                "Recall@100": recall_100
            })

results_df = pd.DataFrame(results)

print(
    results_df.sort_values(
        ["Recall@100", "AP"],
        ascending=False
    ).round(4).to_string(index=False)
)
```

</details>

**Recorded output**

```text
max_depth  min_child_weight  weight_factor     AP  Recall@100
         6                 5            0.5 0.8391      0.8378
         6                 1            0.5 0.8363      0.8378
         6                 5            1.0 0.8302      0.8378
         6                 1            1.0 0.8299      0.8378
         4                 5            0.5 0.8202      0.8378
         4                 1            0.5 0.8173      0.8243
         4                 1            1.0 0.8163      0.8243
         4                 5            1.0 0.8147      0.8243
         3                 1            0.5 0.7981      0.8243
         3                 5            0.5 0.7911      0.8243
         3                 5            1.0 0.7721      0.8243
         3                 1            1.0 0.7666      0.8243
```

### Original versus engineered feature sets

<details><summary>Original experiment code (click to expand)</summary>

```python
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

feature_sets = {
    "Original": list(X_train.columns),
    "Original + LogAmount": list(X_train.columns) + ["LogAmount"],
    "Original + LogAmount + PCA_Norm":
        list(X_train.columns) + ["LogAmount", "PCA_Norm"],
    "All engineered": list(train_fe.columns)
}

results = []

for name, columns in feature_sets.items():

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        min_child_weight=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=imbalance_ratio * 0.5,
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1
    )

    model.fit(train_fe[columns], y_train)

    scores = model.predict_proba(val_fe[columns])[:, 1]

    top100 = np.argsort(scores)[-100:]
    frauds_detected = int(np.asarray(y_val)[top100].sum())

    results.append({
        "Features": name,
        "AP": average_precision_score(y_val, scores),
        "Frauds@100": frauds_detected,
        "Recall@100": frauds_detected / np.asarray(y_val).sum()
    })

results_df = pd.DataFrame(results)

print(results_df.round(4).to_string(index=False))
```

</details>

**Recorded output**

```text
Features     AP  Frauds@100  Recall@100
                       Original 0.8357          63      0.8514
           Original + LogAmount 0.8369          62      0.8378
Original + LogAmount + PCA_Norm 0.8298          62      0.8378
                 All engineered 0.8391          62      0.8378
```

**Selection note:** The original features had the best recorded Frauds@100 (63) among these feature sets, even though the fully engineered feature set had a slightly higher validation AP.

## 6. Regularization

### L1, L2 and gamma comparison

<details><summary>Original experiment code (click to expand)</summary>

```python
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import average_precision_score

configs = [
    {"name": "Baseline", "reg_alpha": 0, "reg_lambda": 1, "gamma": 0},
    {"name": "L1", "reg_alpha": 1, "reg_lambda": 1, "gamma": 0},
    {"name": "L2", "reg_alpha": 0, "reg_lambda": 5, "gamma": 0},
    {"name": "L1 + L2 + Gamma", "reg_alpha": 1, "reg_lambda": 5, "gamma": 1}
]

results = []
models = {}

for config in configs:
    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        min_child_weight=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=imbalance_ratio * 0.5,
        reg_alpha=config["reg_alpha"],
        reg_lambda=config["reg_lambda"],
        gamma=config["gamma"],
        objective="binary:logistic",
        eval_metric="aucpr",
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train)

    scores = model.predict_proba(X_val)[:, 1]
    top100 = np.argsort(scores)[-100:]
    detected = int(np.asarray(y_val)[top100].sum())

    results.append({
        "Model": config["name"],
        "AP": average_precision_score(y_val, scores),
        "Frauds@100": detected,
        "Recall@100": detected / np.asarray(y_val).sum()
    })

    models[config["name"]] = model

print(pd.DataFrame(results).round(4).to_string(index=False))
```

</details>

**Recorded output**

```text
Model     AP  Frauds@100  Recall@100
       Baseline 0.8357          63      0.8514
             L1 0.8333          62      0.8378
             L2 0.8320          62      0.8378
L1 + L2 + Gamma 0.8323          62      0.8378
```

## 7. LightGBM benchmark

### LightGBM compared with XGBoost

<details><summary>Original experiment code (click to expand)</summary>

```python
import numpy as np
from sklearn.metrics import average_precision_score

y = np.asarray(y_val)

lgbm_scores = lgbm_model.predict_proba(val_fe)[:, 1]
xgb_scores = best_xgb.predict_proba(val_fe)[:, 1]

def top_100(scores):
    selected = np.zeros(len(scores), dtype=bool)
    selected[np.argsort(scores)[-100:]] = True
    return selected

lgbm_top = top_100(lgbm_scores)
xgb_top = top_100(xgb_scores)

fraud = y == 1

both = fraud & lgbm_top & xgb_top
only_lgbm = fraud & lgbm_top & ~xgb_top
only_xgb = fraud & ~lgbm_top & xgb_top
neither = fraud & ~lgbm_top & ~xgb_top

print("LightGBM AP:",
      round(average_precision_score(y, lgbm_scores), 4))
print("XGBoost AP:",
      round(average_precision_score(y, xgb_scores), 4))

print("\nLightGBM Recall@100:",
      round(y[lgbm_top].sum() / fraud.sum(), 4))
print("XGBoost Recall@100:",
      round(y[xgb_top].sum() / fraud.sum(), 4))

print("\nDetected by both:", both.sum())
print("Only LightGBM:", only_lgbm.sum())
print("Only XGBoost:", only_xgb.sum())
print("Missed by both:", neither.sum())
```

</details>

**Recorded output**

```text
LightGBM AP: 0.1923
XGBoost AP: 0.8391

LightGBM Recall@100: 0.3514
XGBoost Recall@100: 0.8378

Detected by both: 25
Only LightGBM: 1
Only XGBoost: 37
Missed by both: 11
```

## 8. Error analysis

### Overlap in missed and detected frauds

<details><summary>Original experiment code (click to expand)</summary>

```python
import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score

# 1. Get fraud probabilities from both models
lr_scores = engineered_model.predict_proba(val_fe)[:, 1]
xgb_scores = best_xgb.predict_proba(val_fe)[:, 1]

y = np.asarray(y_val)

# 2. Select top 100 highest-risk transactions
def top_100(scores):
    selected = np.zeros(len(scores), dtype=bool)
    selected[np.argsort(scores)[-100:]] = True
    return selected

lr_detected = top_100(lr_scores)
xgb_detected = top_100(xgb_scores)

# 3. Compare actual fraudulent transactions
fraud = y == 1

both = fraud & lr_detected & xgb_detected
only_lr = fraud & lr_detected & ~xgb_detected
only_xgb = fraud & ~lr_detected & xgb_detected
neither = fraud & ~lr_detected & ~xgb_detected

# 4. Print comparison
print("Detected by both:", both.sum())
print("Detected only by Logistic Regression:", only_lr.sum())
print("Detected only by XGBoost:", only_xgb.sum())
print("Missed by both:", neither.sum())

print("\nLogistic Regression AP:",
      round(average_precision_score(y, lr_scores), 4))

print("XGBoost AP:",
      round(average_precision_score(y, xgb_scores), 4))

# 5. Inspect frauds LR catches but XGBoost misses
recovered = val_fe.iloc[np.where(only_lr)[0]].copy()

recovered["LR_Score"] = lr_scores[only_lr]
recovered["XGB_Score"] = xgb_scores[only_lr]

print("\nFrauds detected only by Logistic Regression:")

print(
    recovered[
        ["Amount", "V14", "V10",
         "PCA_Norm", "LR_Score", "XGB_Score"]
    ].round(4).to_string()
)
```

</details>

**Recorded output**

```text
Detected by both: 60
Detected only by Logistic Regression: 0
Detected only by XGBoost: 2
Missed by both: 12

Logistic Regression AP: 0.6697
XGBoost AP: 0.8391

Frauds detected only by Logistic Regression:
Empty DataFrame
Columns: [Amount, V14, V10, PCA_Norm, LR_Score, XGB_Score]
Index: []
```

## 9. Research notebook final test result

### Test evaluation on the original stratified split

<details><summary>Original experiment code (click to expand)</summary>

```python
import numpy as np
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

final_model = models["Baseline"]

# Predict on untouched test data
test_scores = final_model.predict_proba(X_test)[:, 1]

# Standard threshold evaluation
test_predictions = (test_scores >= 0.5).astype(int)

print("FINAL TEST RESULTS")
print("----------------------------")
print("Average Precision:",
      round(average_precision_score(y_test, test_scores), 4))

print("ROC-AUC:",
      round(roc_auc_score(y_test, test_scores), 4))

print("Precision:",
      round(precision_score(y_test, test_predictions, zero_division=0), 4))

print("Recall:",
      round(recall_score(y_test, test_predictions, zero_division=0), 4))

print("F1:",
      round(f1_score(y_test, test_predictions, zero_division=0), 4))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, test_predictions))

# Top 100 highest-risk transactions
y_test_array = np.asarray(y_test)

top100_indices = np.argsort(test_scores)[-100:]
detected = int(y_test_array[top100_indices].sum())
total_frauds = int(y_test_array.sum())

print("\nTOP 100 INVESTIGATION RESULTS")
print("----------------------------")
print("Total test frauds:", total_frauds)
print("Frauds detected:", detected)
print("False alarms:", 100 - detected)
print("Frauds missed:", total_frauds - detected)
print("Recall@100:", round(detected / total_frauds, 4))
print("Precision@100:", round(detected / 100, 4))
```

</details>

**Recorded output**

```text
FINAL TEST RESULTS
----------------------------
Average Precision: 0.8347
ROC-AUC: 0.9679
Precision: 0.8571
Recall: 0.8108
F1: 0.8333

Confusion Matrix:
[[42638    10]
 [   14    60]]

TOP 100 INVESTIGATION RESULTS
----------------------------
Total test frauds: 74
Frauds detected: 62
False alarms: 38
Frauds missed: 12
Recall@100: 0.8378
Precision@100: 0.62
```

## Conclusion

The recorded research led to a cost-sensitive XGBoost classifier using the original 30 features. The research notebook's stratified-split test AP was **0.8347**, with **62 of 74** test frauds found in the top 100 alerts. These are **research-run results**, not the simplified chronological notebook's results (**AP 0.7692**, **40 of 52** frauds found in its top 100). Both are retained with their split definitions for transparency.

**Reproducibility:** This curated notebook is an experiment report, not a linear executable notebook. Code snippets depend on objects created during the original exploratory session; use the separate final notebook for end-to-end execution. Some exploratory trials (including external experiment-tracking logs) were omitted for readability.
